# 01 – Parse & Clean NASA HTTP Logs

This notebook:
1. Loads the sample log file shipped with the repo (`data/sample_logs.txt`).
2. Parses it into a structured DataFrame using `pipeline.log_parser`.
3. Applies basic cleaning.
4. Saves the result to `data/processed/parsed_logs.parquet`.

In [1]:
import sys, pathlib
# Ensure repo root is on the path when running from notebooks/
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))


In [2]:
import pandas as pd
from pipeline.log_parser import parse_log_file

LOG_FILE = ROOT / "data" / "sample_logs.txt"
print(f"Parsing {LOG_FILE} …")
df = parse_log_file(str(LOG_FILE))
print(f"Parsed {len(df):,} log entries.")
df.head()


Parsing /home/runner/work/log/log/data/sample_logs.txt …
Parsed 760 log entries.


,ip_address,timestamp,method,endpoint,protocol,status_code,bytes_sent,referer,user_agent
0,192.168.99.1,2024-01-15 03:00:07+00:00,POST,/login,HTTP/1.1,403,292,-,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
1,192.168.99.1,2024-01-15 03:00:26+00:00,POST,/login,HTTP/1.1,403,56,-,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
2,192.168.99.1,2024-01-15 03:00:48+00:00,POST,/login,HTTP/1.1,401,249,-,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
3,192.168.99.1,2024-01-15 03:01:29+00:00,POST,/login,HTTP/1.1,200,100,-,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...
4,192.168.99.1,2024-01-15 03:01:46+00:00,POST,/login,HTTP/1.1,401,106,-,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...


In [3]:
# ── Basic cleaning ────────────────────────────────────────────────────────────
# 1. Drop rows with a null timestamp (malformed lines)
before = len(df)
df = df.dropna(subset=["timestamp"])
print(f"Dropped {before - len(df)} rows with null timestamps.")

# 2. Filter out HEAD/OPTIONS noise
df = df[~df["method"].isin(["HEAD", "OPTIONS"])].copy()
print(f"Rows after method filter: {len(df):,}")

# 3. Normalise endpoint (strip query string)
df["endpoint"] = df["endpoint"].str.split("?").str[0]

# 4. Clip absurdly large byte values (> 10 MB) to NaN then fill with median
BYTES_MAX = 10_000_000
df.loc[df["bytes_sent"] > BYTES_MAX, "bytes_sent"] = df["bytes_sent"].median()

df.dtypes


Dropped 0 rows with null timestamps.
Rows after method filter: 760


ip_address                     str
timestamp      datetime64[us, UTC]
method                         str
endpoint                    object
protocol                       str
status_code                  int64
bytes_sent                   int64
referer                        str
user_agent                     str
dtype: object

In [4]:
# ── Save processed dataset ────────────────────────────────────────────────────
OUT = ROOT / "data" / "processed" / "parsed_logs.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(str(OUT), index=False)
print(f"Saved → {OUT}  ({len(df):,} rows)")


Saved → /home/runner/work/log/log/data/processed/parsed_logs.parquet  (760 rows)


In [5]:
# ── Quick sanity check ────────────────────────────────────────────────────────
print("Columns:", df.columns.tolist())
print("\nStatus code distribution:")
print(df["status_code"].value_counts().head(10))
print("\nTop 5 IPs by request count:")
print(df["ip_address"].value_counts().head(5))


Columns: ['ip_address', 'timestamp', 'method', 'endpoint', 'protocol', 'status_code', 'bytes_sent', 'referer', 'user_agent']

Status code distribution:
status_code
200    456
401     91
403     75
304     72
404     66
Name: count, dtype: int64

Top 5 IPs by request count:
ip_address
192.168.99.1    150
45.33.32.156     80
10.0.0.18        36
10.0.0.8         32
10.0.0.16        32
Name: count, dtype: int64
